In [1]:
!pip install turboquant

In [3]:
import gradio as gr
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from turboquant import TurboQuantCache

# 1. Initialize Model & Cache Globally (Loads once)
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Initialize the 4-bit compressed KV cache
compressed_cache = TurboQuantCache(bits=4)

# 2. Define the Chat Function
def generate_response(message, history):
    # Construct the message history
    messages = [{"role": "system", "content": "You are a concise coding assistant."}]

    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})

    messages.append({"role": "user", "content": message})

    # Tokenize inputs
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    # Generate output
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        past_key_values=compressed_cache,
        use_cache=True
    )

    # Decode and return the new tokens
    input_length = inputs.input_ids.shape[1]
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

    return response

# 3. Build and Launch the UI
app = gr.ChatInterface(
    fn=generate_response,
    title="TurboQuant LLM Chat",
    description="Chatting locally with a 1.5B model using a highly compressed 4-bit KV cache.",
)

if __name__ == "__main__":
    app.launch()

Loading Qwen/Qwen2.5-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://567231557a346688e8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
